# Agentic Search with Database Query Tool

This notebook walks you through an agentic search example with a general-purpose search tool to execute ES|QL queries against an Elasticsearch index. 
It improves the first implementation by adding Agent Skills for writing ES|QL queries.

This approach lets the agent write entire search queries from scratch to pull context stored in a database into its context window to answer more **complex** user queries.

<img src="../img/Agentic search with DB tool.png" width="1000" alt="Agentic search with db query tool" />

Note, this is applicable to other query languages, such as SQL as well.

## Preparation

Make sure you have completed the setup instructions according to the README.md.

Load environment variables from `.env` file, such as API keys.

In [1]:
import os                                                                                                                                                                                                          
from dotenv import load_dotenv

load_dotenv(override=True)

LITELLM_API_KEY=os.getenv("LITELLM_API_KEY")
LITELLM_API_BASE=os.getenv("LITELLM_API_BASE")

ES_URL=os.getenv("ELASTICSEARCH_URL")
ES_USERNAME=os.getenv("ELASTICSEARCH_USERNAME")
ES_PASSWORD=os.getenv("ELASTICSEARCH_PASSWORD")

## Set up agent with a database query tool

### Configure the LLM

Set up your LLM with the right parameters for your use case:
Below, we're using OpenAI's models through LiteLLM but you can switch it out with any model capable of tool use.

Note: For this notebook, we had to switch from `gpt-5.4-nano` to `gpt-5.4-mini` for better performance.

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_base=LITELLM_API_BASE,
    api_key=LITELLM_API_KEY,
    model="llm-gateway/gpt-5.4-mini",
    temperature=0.5,
)

/Users/leonie/Documents/code/workshop-agentic-search/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


### Define the system prompt

The system prompt defines your agent’s role and behavior. 

We also include information about what out-of-context knowledge source it has. In this case it has access to an **Elasticsearch index** with the conference data.

In [3]:
from IPython.display import display, Markdown

with open("system_prompt_db.md", 'r') as f:
    SYSTEM_PROMPT = f.read()

display(Markdown(SYSTEM_PROMPT))

You are a search agent tasked with answering questions about the AI Engineer Europe 2026 Conference.

You have access to different context retrieval tools to help you answer user queries.

Before answering a question decide whether or not you need to retrieve additional context to answer the question correctly.
If the retrieved context does not contain relevant information to answer the query, say that you don't know. 

## Elasticsearch (`conference_schedule`)

The conference sessions are indexed in Elasticsearch. One document per session.

| Field | Description |
|--------------|------------|
| `text` | The string that was embedded: each session’s title plus description (blank line between them). It does not include day, time, room, or speakers. |
| `vector` | Dense embedding of `text`, used for similarity search. |
| `metadata` | Structured fields (see metadata description below) |


| Field | Description |
|--------------|------------|
| `metadata.title` | Title of the session|
| `metadata.day` | Date of the session (Example format: April 10) |
| `metadata.time` | Time slot of the session (Example format: 12:40–1:00pm) |
| `metadata.room` | Room where the session takes place |
| `metadata.type` | One of 'keynote', 'workshop', 'talk', 'track_keynote', 'lightning', 'expo_session' |
| `metadata.track` | Track |
| `metadata.speakers` | Name(s) of the speaker(s) as a single comma-separated string |


### Create general-purpose database query tool

Let's create a general-purpose search tool called `execute_esql_query` that lets the agent write an ES|QL query from scratch and execute it against the Elasticsearch index.

<div class="alert alert-block alert-info">
<b>What is ES|QL?</b>

Elasticsearch Query Language (ES|QL) is a piped query language for filtering, transforming, and analyzing data.

Learn more about ES|QL in the <a href="https://www.elastic.co/docs/reference/query-languages/esql" target="_blank" rel="noopener noreferrer">ES|QL reference</a>.
</div>

In [4]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(
    hosts=ES_URL,
    basic_auth=(ES_USERNAME, ES_PASSWORD)
)

# Test the ES|QL execution
response = es_client.esql.query(
    query='''FROM conference_schedule 
    | WHERE text LIKE "*GEPA*"
    | KEEP metadata.speakers, metadata.title, text
    | LIMIT 3
    ''',
    format="csv",
)

print(response.body)

metadata.speakers,metadata.title,text
Samuel Colvin,Playground in Prod - Optimising Agents in Production Environments,"Playground in Prod - Optimising Agents in Production Environments

Deploying an agent is just the beginning. The real challenge is making it better once it's live — without redeploying, without downtime, and ideally without a human in the loop at all.

In this talk I'll introduce two complementary approaches to optimising agents in production, both built on Pydantic AI and Logfire:

**Managed Variables** — a new Pydantic AI feature (build on the Open Feature standard) that lets you externalise key parameters of your agent (system prompts, model configuration, tool descriptions, thresholds) and update them instantly from the Logfire UI. Change a system prompt, swap a model, or adjust a temperature parameter and see the effect on the next request — no redeploy, no restart. This turns production into a playground where you can iterate on agent behaviour in seconds based o

Convert the Python function to a tool using the [`@tool` decorator](https://reference.langchain.com/python/langchain-core/tools/convert/tool). By default the function name becomes the **tool name** and the docstring becomes the **tool description**.

In [5]:
from langchain.tools import tool

@tool()
def execute_esql_query(esql_query: str) -> str:
    """Execute an ES|QL query against the conference_schedule index in Elasticsearch.

    Args:
        esql_query: The ES|QL query to execute

    Returns:
        A string containing the information about the sessions.
    """
    try:
        response = es_client.esql.query(
            query=esql_query,
            format="csv",
        )
        return response.body 

    except Exception as e:
        return f"Error executing ES|QL query: {e}"

# Test the query execution tool
esql_query = '''FROM conference_schedule 
    | WHERE text LIKE "*GEPA*"
    | KEEP metadata.speakers, metadata.title, text
    | LIMIT 3'''

print(execute_esql_query.invoke({"esql_query": esql_query}))

metadata.speakers,metadata.title,text
Samuel Colvin,Playground in Prod - Optimising Agents in Production Environments,"Playground in Prod - Optimising Agents in Production Environments

Deploying an agent is just the beginning. The real challenge is making it better once it's live — without redeploying, without downtime, and ideally without a human in the loop at all.

In this talk I'll introduce two complementary approaches to optimising agents in production, both built on Pydantic AI and Logfire:

**Managed Variables** — a new Pydantic AI feature (build on the Open Feature standard) that lets you externalise key parameters of your agent (system prompts, model configuration, tool descriptions, thresholds) and update them instantly from the Logfire UI. Change a system prompt, swap a model, or adjust a temperature parameter and see the effect on the next request — no redeploy, no restart. This turns production into a playground where you can iterate on agent behaviour in seconds based o

### Create the agent

Now assemble your agent with all the components including the general-purpose search tool (`execute_esql_query`).

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[execute_esql_query],
)

## Run the agent

### Example: Agent uses tool to retrieve context

In [7]:
input_message = {
    "role": "user",
    "content": ("Which sessions should I visit to learn more about GEPA?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions should I visit to learn more about GEPA?
================================== Ai Message ==================================
Tool Calls:
  execute_esql_query (call_Tg3DnDlcJ4eZwqI7smNFHpH6)
 Call ID: call_Tg3DnDlcJ4eZwqI7smNFHpH6
  Args:
    esql_query: FROM conference_schedule
| WHERE text LIKE "%GEPA%" OR metadata.title LIKE "%GEPA%"
| KEEP metadata.title, metadata.day, metadata.time, metadata.room, metadata.type, metadata.track, metadata.speakers
| SORT metadata.day, metadata.time
================================= Tool Message =================================
Name: execute_esql_query

metadata.title,metadata.day,metadata.time,metadata.room,metadata.type,metadata.track,metadata.speakers



/var/folders/64/w4vvrnpx0nxf4rnr04bc39lc0000gn/T/ipykernel_42834/882490044.py:14: ElasticsearchWarning: No limit defined, adding default limit of [1000]
  response = es_client.esql.query(


================================== Ai Message ==================================

I couldn’t find any sessions that mention GEPA in the conference schedule. So I don’t know of any session you should visit specifically to learn about it.


**Problem:** LLM generates incorrect ES|QL query (in this example ES|QL doesn't use the percentage sign (`%`) as a wildcard character but an asteriks (`*`)).

**Solution idea:** Use an Agent Skill for writing ES|QL queries

## Add an Agent Skill and a skill tool 

Let's add an Agent Skill to help the agent write better ES|QL queries and implement a [skill loading tool](https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant).

### Define Skill

For brevity and demo-purposes, we will write a short ES|QL Skill ourselves. 

Alternatively, you can install official [Elastic Agent Skills](https://github.com/elastic/agent-skills/tree/main) via:

```bash
npx skills add elastic/agent-skills
```

In [8]:
from typing import TypedDict

class Skill(TypedDict):
    """A skill that can be progressively disclosed to the agent."""
    name: str  # Unique identifier for the skill
    description: str  # 1-2 sentence description to show in system prompt
    content: str  # Full skill content with detailed instructions

SKILLS: list[Skill] = [
    {
        "name": "elasticsearch-esql",
        "description": """Execute ES|QL (Elasticsearch Query Language) queries, use when the user wants to
  query Elasticsearch data, analyze logs, aggregate metrics, explore data, or create
  charts and dashboards from ES|QL results.""",
        "content": """
# Elasticsearch ES|QL

Execute ES|QL queries against Elasticsearch.

## What is ES|QL?

ES|QL (Elasticsearch Query Language) is a piped query language for Elasticsearch. It is **NOT** the same as SQL.

ES|QL uses pipes (`|`) to chain commands:
`FROM index | WHERE condition | STATS aggregation BY field | SORT field | LIMIT n`

## Critical Syntax Rules

### String Literals Use Double Quotes Only

ES|QL uses **double quotes** for string literals — never single quotes. This is the most common source of
`token recognition error at: '` failures. SQL habits lead models to write `'value'` when ES|QL requires `"value"`.

### Comparison Operators

- `==` Equal
- `!=` Not equal
- `<`, `<=`, `>`, `>=` Comparison
- `IS NULL`, `IS NOT NULL` Null checks

### Logical Operators

- `AND` Logical AND
- `OR` Logical OR
- `NOT` Logical NOT

### Pattern Matching

- `LIKE` Wildcard pattern (`*` zero or more chars, `?` single char)
- `RLIKE` Regular expression
- `IN` Value in list
"""
    },
]

### Create skill loading tool

Create a tool to load full skill content on-demand.

In [9]:
from langchain.tools import tool

@tool
def load_skill(skill_name: str) -> str:
    """Load the full content of a skill into the agent's context.

    Use this when you need detailed information about how to handle a specific
    type of request. This will provide you with comprehensive instructions,
    policies, and guidelines for the skill area.

    Args:
        skill_name: The name of the skill to load (e.g., "elasticsearch-esql")
    """
    # Find and return the requested skill
    for skill in SKILLS:
        if skill["name"] == skill_name:
            return f"Loaded skill: {skill_name}\n\n{skill['content']}"

    # Skill not found
    available = ", ".join(s["name"] for s in SKILLS)
    return f"Skill '{skill_name}' not found. Available skills: {available}"

### Build skill middleware

Create custom middleware that injects skill descriptions into the system prompt. This middleware makes skills discoverable without loading their full content upfront.

In [10]:
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware
from langchain.messages import SystemMessage
from typing import Callable

class SkillMiddleware(AgentMiddleware):
    """Middleware that injects skill descriptions into the system prompt."""

    # Register the load_skill tool as a class variable
    tools = [load_skill]

    def __init__(self):
        """Initialize and generate the skills prompt from SKILLS."""
        # Build skills prompt from the SKILLS list
        skills_list = []
        for skill in SKILLS:
            skills_list.append(
                f"- **{skill['name']}**: {skill['description']}"
            )
        self.skills_prompt = "\n".join(skills_list)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """Sync: Inject skill descriptions into system prompt."""
        # Build the skills addendum
        skills_addendum = (
            f"\n\n## Available Skills\n\n{self.skills_prompt}\n\n"
            "Use the load_skill tool when you need detailed information "
            "about handling a specific type of request."
        )

        # Append to system message content blocks
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": skills_addendum}
        ]
        new_system_message = SystemMessage(content=new_content)
        modified_request = request.override(system_message=new_system_message)
        return handler(modified_request)

### Edit the tool description of the general-purpose database query tool

Because we want the agent to first load the ES|QL skill to generate an ES|QL query before it calls te execution tool, we will adjust the tool description to reflect this relationship by adding `"Always use the Elasticsearch ES|QL skill to generate the ES|QL query before using this tool to execute the query."`

In [11]:
from langchain.tools import tool

@tool()
def execute_esql_query(esql_query: str) -> str:
    """Execute an ES|QL query against and index in Elasticsearch.
    Always use the Elasticsearch ES|QL skill to generate the ES|QL query before using this tool to execute the query.

    Args:
        esql_query: The ES|QL query to execute

    Returns:
        A string containing the information about the sessions.
    """
    try:
        response = es_client.esql.query(
            query=esql_query,
            format="csv",
        )
        return response.body 

    except Exception as e:
        return f"Error executing ES|QL query: {e}"

### Edit the system prompt

We will also reinforce the relationship of calling the ES|QL skill before calling the execution tool in the system prompt.

In [12]:
SYSTEM_PROMPT_WITH_SKILL = SYSTEM_PROMPT +  "If you need to execute an ES|QL query, use the Elasticsearch ES|QL skill to generate the query before using this tool to execute the query. If an ESQL query returns an error use the Elasticsearch ES|QL skill to generate a new query."

### Create the agent

Now assemble your agent with all the components including the general-purpose search tool (`execute_esql_query`) and the skill middleware.

In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT_WITH_SKILL,
    middleware=[SkillMiddleware()],
    tools=[execute_esql_query],
)

## Run the agent

### Example: Agent generates a full-text search query

In [14]:
input_message = {
    "role": "user",
    "content": ("Which sessions should I visit to learn more about GEPA?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions should I visit to learn more about GEPA?
================================== Ai Message ==================================
Tool Calls:
  load_skill (call_pUblqD2S7iLb6hjEaI7xZZJr)
 Call ID: call_pUblqD2S7iLb6hjEaI7xZZJr
  Args:
    skill_name: elasticsearch-esql
================================= Tool Message =================================
Name: load_skill

Loaded skill: elasticsearch-esql


# Elasticsearch ES|QL

Execute ES|QL queries against Elasticsearch.

## What is ES|QL?

ES|QL (Elasticsearch Query Language) is a piped query language for Elasticsearch. It is **NOT** the same as SQL.

ES|QL uses pipes (`|`) to chain commands:
`FROM index | WHERE condition | STATS aggregation BY field | SORT field | LIMIT n`

## Critical Syntax Rules

### String Literals Use Double Quotes Only

ES|QL uses **double quotes** for string literals — never single quotes. This is the most common source of
`to

/var/folders/64/w4vvrnpx0nxf4rnr04bc39lc0000gn/T/ipykernel_42834/1149930936.py:15: ElasticsearchWarning: No limit defined, adding default limit of [1000]
  response = es_client.esql.query(


================================== Ai Message ==================================

The session I found related to GEPA is:

- **Playground in Prod - Optimising Agents in Production Environments**  
  - **When:** April 8, 10:40am–12:00pm  
  - **Where:** Westminster  
  - **Type:** Workshop  
  - **Speaker:** Samuel Colvin

If you want, I can also look for sessions about **agents**, **optimization**, or **production environments** that may be relevant to GEPA even if they don’t mention it by name.


### Example: Agent generates an aggregation query

In [15]:
input_message = {
    "role": "user",
    "content": ("How many sessions are on April 8th?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How many sessions are on April 8th?
================================== Ai Message ==================================
Tool Calls:
  load_skill (call_5Ki25LYVnkx245nEDSM3xwwz)
 Call ID: call_5Ki25LYVnkx245nEDSM3xwwz
  Args:
    skill_name: elasticsearch-esql
================================= Tool Message =================================
Name: load_skill

Loaded skill: elasticsearch-esql


# Elasticsearch ES|QL

Execute ES|QL queries against Elasticsearch.

## What is ES|QL?

ES|QL (Elasticsearch Query Language) is a piped query language for Elasticsearch. It is **NOT** the same as SQL.

ES|QL uses pipes (`|`) to chain commands:
`FROM index | WHERE condition | STATS aggregation BY field | SORT field | LIMIT n`

## Critical Syntax Rules

### String Literals Use Double Quotes Only

ES|QL uses **double quotes** for string literals — never single quotes. This is the most common source of
`token recognition erro

/var/folders/64/w4vvrnpx0nxf4rnr04bc39lc0000gn/T/ipykernel_42834/1149930936.py:15: ElasticsearchWarning: No limit defined, adding default limit of [1000]
  response = es_client.esql.query(


================================== Ai Message ==================================

There are **27 sessions** on **April 8th**.
